In [ ]:
import arxiv
from datetime import datetime, timedelta, timezone
import pprint
from rich import inspect
import pickle
import json
from fastembed import TextEmbedding
import os
from dotenv import load_dotenv
from clients.llm_client import LLMClient

In [10]:
load_dotenv()

True

In [ ]:
llm_client = LLMClient(os.environ["GEMINI_API_KEY"])

In [11]:
# # 1. Calculate time in UTC, as explicitly required by the ArXiv API
# end_date = datetime.now(timezone.utc)
# start_date = end_date - timedelta(days=30)

# # 2. Format to YYYYMMDDHHMM (ArXiv's "TTTT" is hours and minutes in 24h format)
# start_str = start_date.strftime("%Y%m%d%H%M")
# end_str = end_date.strftime("%Y%m%d%H%M")

# # 3. Construct the exact Lucene query string
# query_string = f"cat:cs.AI AND submittedDate:[{start_str} TO {end_str}]"

# # 4. Initialize client and search parameters
# client = arxiv.Client()
# search = arxiv.Search(
#     query=query_string,
#     max_results=100,
#     sort_by=arxiv.SortCriterion.SubmittedDate,
#     sort_order=arxiv.SortOrder.Descending
# )

In [12]:
# results = list(client.results(search))

In [13]:
# # Assuming 'results_list' is your materialized list from client.results(search)
# points = []
# for result in results:
#     points.append({
#         'title': result.title,
#         'categories': result.categories, # Already a list of strings
#         'authors': [author.name for author in result.authors], # Extract string names
#         'summary': result.summary,
#         'entry_id': result.entry_id,
#         'published': result.published.isoformat() # Convert datetime to string
#     })

# # Write to a local file
# with open('arxiv_data_cache.json', 'w', encoding='utf-8') as file:
#     json.dump(points, file, indent=4)

In [14]:
with open('arxiv_data_cache.json', 'r', encoding='utf-8') as file:
    points = json.load(file)

In [15]:
pprint.pprint((points[0]))

{'authors': ['Eric Gan',
             'Aryan Bhatt',
             'Buck Shlegeris',
             'Julian Stastny',
             'Vivek Hebbar'],
 'categories': ['cs.AI'],
 'entry_id': 'http://arxiv.org/abs/2604.16286v1',
 'published': '2026-04-17T17:47:32+00:00',
 'summary': 'As AI systems are increasingly used to conduct research '
            'autonomously, misaligned systems could introduce subtle flaws '
            'that produce misleading results while evading detection. We '
            'introduce ASMR-Bench (Auditing for Sabotage in ML Research), a '
            'benchmark for evaluating the ability of auditors to detect '
            'sabotage in ML research codebases. ASMR-Bench consists of 9 ML '
            'research codebases with sabotaged variants that produce '
            'qualitatively different experimental results. Each sabotage '
            'modifies implementation details, such as hyperparameters, '
            'training data, or evaluation code, while preserving

In [16]:
from qdrant_client import QdrantClient, models

client = QdrantClient("http://localhost:6333")
client.get_collections()

CollectionsResponse(collections=[CollectionDescription(name='arxiv-rag')])

In [17]:
EMBEDDING_DIMENSIONALITY = 512
model_handle = "jinaai/jina-embeddings-v2-small-en"

In [18]:
collection_name = "arxiv-rag"

# Create the collection with specified vector parameters
# client.create_collection(
#     collection_name=collection_name,
#     vectors_config=models.VectorParams(
#         size=EMBEDDING_DIMENSIONALITY,  # Dimensionality of the vectors
#         distance=models.Distance.COSINE  # Distance metric for similarity search
#     )
# )

In [19]:
id = 0
qdrant_points = []

for point in points:
    entry = models.PointStruct(
        id=id,
        vector=models.Document(text=point['summary'], model=model_handle),
        payload={
            **point,
            'id': id
        } #save all needed metadata fields
    )
    qdrant_points.append(entry)

    id += 1

In [20]:
print(qdrant_points[0].payload)
# print(qdrant_points[0])

{'title': 'ASMR-Bench: Auditing for Sabotage in ML Research', 'categories': ['cs.AI'], 'authors': ['Eric Gan', 'Aryan Bhatt', 'Buck Shlegeris', 'Julian Stastny', 'Vivek Hebbar'], 'summary': 'As AI systems are increasingly used to conduct research autonomously, misaligned systems could introduce subtle flaws that produce misleading results while evading detection. We introduce ASMR-Bench (Auditing for Sabotage in ML Research), a benchmark for evaluating the ability of auditors to detect sabotage in ML research codebases. ASMR-Bench consists of 9 ML research codebases with sabotaged variants that produce qualitatively different experimental results. Each sabotage modifies implementation details, such as hyperparameters, training data, or evaluation code, while preserving the high-level methodology described in the paper. We evaluated frontier LLMs and LLM-assisted human auditors on ASMR-Bench and found that both struggled to reliably detect sabotage: the best performance was an AUROC of 

In [21]:
client.upsert(
    collection_name=collection_name,
    points=qdrant_points
)

UpdateResult(operation_id=7, status=<UpdateStatus.COMPLETED: 'completed'>)

In [22]:
query = 'What are the latest happenings in the alignment and safety field?'

In [23]:
def search(query, limit=3):
    result = client.query_points(
        collection_name=collection_name,
        query=models.Document(
            text=query,
            model=model_handle
        ),
        limit=limit,
        with_payload=True
    )

    return result

In [24]:
result = search(query)
print(result.points[0])

id=77 version=7 score=0.7633171 payload={'title': 'The World Leaks the Future: Harness Evolution for Future Prediction Agents', 'categories': ['cs.AI'], 'authors': ['Chuyang Wei', 'Maohang Gao', 'Zhixin Han', 'Kefei Chen', 'Yu Zhuang', 'Haoxiang Guan', 'Yanzhi Zhang', 'Yilin Cheng', 'Jiyan He', 'Huanhuan Chen', 'Jian Li', 'Yu Shi', 'Yitong Duan', 'Shuxin Zheng'], 'summary': 'Many consequential decisions must be made before the relevant outcome is known. Such problems are commonly framed as \\emph{future prediction}, where an LLM agent must form a prediction for an unresolved question using only the public information available at the prediction time. The setting is difficult because public evidence evolves while useful supervision arrives only after the question is resolved, so most existing approaches still improve mainly from final outcomes. Yet final outcomes are too coarse to guide earlier factor tracking, evidence gathering and interpretation, or uncertainty handling. When the sam

In [25]:
print(len(result.points))

3


In [ ]:
def build_prompt(query_results, query) -> str:
    system_prompt = (
        "You are a chatbot designed to synthesize and summarize the results of a search "
        "on the Arxiv database of papers in the last 30 days based on the user's queries. "
        f"QUERY: {query}\n\n"
        "Based on the query, the search returned these documents that might be relevant. Review the following retrieved documents carefully:\n\n"
    )

    formatted_results = []
    iter = len(query_results.points)
    for i in range(0, iter):
        # Dynamically handle an arbitrary number of authors using .join()
        authors_list = query_results.points[i].payload.get("authors", ["Unknown Author"])
        authors_str = ", ".join(authors_list)
        categories_list = query_results.points[i].payload.get("categories", ["Unknown category"])
        categories_str = ", ".join(categories_list)
        title = query_results.points[i].payload.get("title", "Untitled")

        # Properly utilize f-strings for variable interpolation
        result_block = (
            f"RESULT {i+1}:\n"
            f"    Title: {title}\n"
            f"    Authors: {authors_str}\n"
            f"    Categories: {categories_str}\n"
            f"    Abstract: {query_results.points[i].payload.get("summary", ["Unknown abstract"])}\n"
            f"    Published date: {query_results.points[i].payload.get("published", ["Unknown date"])}\n"
            f"    Entry ID: {query_results.points[i].payload.get("entry_id", ["Unknown ID"])}\n"
        )
        formatted_results.append(result_block)

    # Assemble the final prompt efficiently
    final_prompt = system_prompt + "\n".join(formatted_results)
    final_prompt_add = final_prompt + "\nNow, answer the user's query."
    return final_prompt_add.strip()

In [27]:
prompt = build_prompt(result, query)

In [28]:
from openai import OpenAI

In [29]:
llm_client = OpenAI(
    api_key=os.environ.get("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)
response = llm_client.chat.completions.create(
    model='gemini-3.1-flash-lite-preview',
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

In [30]:
print(response.choices[0].message.content)

Recent research in AI alignment and safety, as reflected in the latest Arxiv submissions from mid-April 2026, highlights a shift toward more robust, verifiable, and interpretative methodologies. The findings can be synthesized into three key areas:

### 1. Hardened Safety via Symbolic Guardrails
A significant development in agent safety is the move away from reliance on purely neural or training-based methods, which often lack strict reliability guarantees.
*   **Symbolic Guardrails:** Research by Hong et al. (Result 2) advocates for the use of "symbolic guardrails" for domain-specific AI agents. By analyzing 80 state-of-the-art benchmarks, the study found that 85% currently lack concrete safety policies. The authors demonstrate that 74% of policy requirements can be effectively enforced using low-cost symbolic mechanisms, providing stronger security and safety guarantees for high-stakes environments (e.g., finance and healthcare) without compromising agent utility.

### 2. Strategic A